# 08 — Custom CNN with LOSO-CV

`src.models.SmallCNN` (~50k params) on the preprocessed cache. Single-fold
smoke test first, then full LOSO.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (10, 4)

import torch

from src.models import SmallCNN, count_parameters
from src.dataset import WindowDataset
from src.train import train_one_fold, loso_cv, loso_summary


## 1. Load manifest, pick device

In [ ]:
manifest = pd.read_csv(ROOT / "outputs" / "preprocessed" / "manifest.csv")
cache_root = ROOT / "outputs" / "preprocessed"
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {device}")
print(f"windows: {len(manifest)}  subjects: {manifest['subject_id'].nunique()}")
print(f"params: {count_parameters(SmallCNN())}")


## 2. Single-fold sanity check

Hold out one subject; train for a few epochs; check val AUC moves above 0.5.

In [ ]:
hold_out = manifest["subject_id"].unique()[0]
train_subj = [s for s in manifest["subject_id"].unique() if s != hold_out]
tr_ds = WindowDataset(manifest, cache_root, subjects=train_subj, augment=True)
va_ds = WindowDataset(manifest, cache_root, subjects=[hold_out], augment=False)
print(f"hold out {hold_out}:  train={len(tr_ds)}  val={len(va_ds)}")

model = SmallCNN(in_channels=2)
fold = train_one_fold(model, tr_ds, va_ds, epochs=5, batch_size=32,
                     device=device, verbose=True)
print(f"\nbest window-AUC for held-out {hold_out}: {fold['best_val_window_auc']:.3f}")


## 3. Full LOSO-CV

~58 folds. On CPU with `SmallCNN` and ~10 windows/subject, expect roughly
20–60 s per fold → 20–60 min total. Set `EPOCHS` smaller for a quick pass.


In [ ]:
EPOCHS = 15
RUN_FULL = False

if RUN_FULL:
    folds = loso_cv(
        model_factory=lambda: SmallCNN(in_channels=2),
        manifest=manifest, cache_root=cache_root,
        channel_mode="both", epochs=EPOCHS, batch_size=32, device=device,
    )
    print(loso_summary(folds))
    out = ROOT / "outputs" / "metrics" / "cnn_folds.csv"
    folds.to_csv(out, index=False)
    print(f"saved {out}")
else:
    print("set RUN_FULL = True to run all folds")


## 4. Inspect per-fold metrics

In [ ]:
path = ROOT / "outputs" / "metrics" / "cnn_folds.csv"
if path.exists():
    folds = pd.read_csv(path)
    print(folds.describe())
    fig, ax = plt.subplots(figsize=(12, 3))
    colours = folds["subject_label"].map({0: "#4c8", 1: "#e66"})
    ax.bar(folds["subject_id"], folds["subject_prob"], color=colours)
    ax.axhline(0.5, color="k", linestyle="--", alpha=0.5)
    ax.set_ylabel("subject prob(PD)")
    ax.set_title("CNN — LOSO predictions (green=true CTRL, red=true PD)")
    plt.xticks(rotation=90, fontsize=7)
    plt.tight_layout(); plt.show()
else:
    print("run section 3 first")


### Notes
- If LOSO is too slow on CPU, drop `EPOCHS` to 5 for a smoke run, then move to a GPU workstation (GAPS via SSH — see `../project_summary_and_setup.md` Section 6) for the full pass.
- Compare the resulting subject-AUC to Notebook 07's baselines.
